In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("EDA") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .getOrCreate()

tlc = spark.read.parquet("s3a://de300-project7/raw/tlc/")

tlc.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in tlc.columns]).show()

:: loading settings :: url = jar:file:/home/ec2-user/.local/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8c07ae15-94cd-49b5-8433-28b20bcb07a9;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 417ms :: artifacts dl 23ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	----------------------------

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       0|                   0|                    0|        3057123|            0|   3057123|           3057123|           0|    

In [3]:
print("Negative/zero fares:", tlc.filter(F.col("fare_amount") <= 0).count())
print("Zero distance trips:", tlc.filter(F.col("trip_distance") <= 0).count())
print("Zero passengers:", tlc.filter(F.col("passenger_count") == 0).count())
print("Extreme fares >500:", tlc.filter(F.col("fare_amount") > 500).count())
print("Extreme distance >100:", tlc.filter(F.col("trip_distance") > 100).count())

Negative/zero fares: 94364


Zero distance trips: 370198


[Stage 13:======================================>                   (2 + 1) / 3]

Zero passengers: 40638


Extreme fares >500: 162


[Stage 19:======================================>                   (2 + 1) / 3]

Extreme distance >100: 495


In [4]:
tlc.select("fare_amount", "trip_distance", "passenger_count", "total_amount").describe().show()

26/05/20 01:22:04 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 22:======================================>                   (2 + 1) / 3]

+-------+------------------+-----------------+------------------+------------------+
|summary|       fare_amount|    trip_distance|   passenger_count|      total_amount|
+-------+------------------+-----------------+------------------+------------------+
|  count|          11077206|         11077206|           8020083|          11077206|
|   mean|21.249499424326512|6.221664346589067|1.2389285746793393|29.793275276299784|
| stddev|18.735193932290542|620.7149755159404|0.6461159016353886|22.373614697707048|
|    min|           -2555.2|              0.0|                 0|           -2560.2|
|    max|            2555.2|         328522.2|                 9|            2560.2|
+-------+------------------+-----------------+------------------+------------------+

